# 세 구조로 같은 주문을 계산해 봐요

카페라떼 2잔에 학생 할인을 적용해요. A·B·C의 **최종 금액이 모두 7,200원인지** 확인하고, 할인율을 바꿀 위치를 찾아요. 영수증 모양이나 잘못된 입력을 처리하는 방식까지 같은 것은 아니에요.

실행 1과 실행 2는 **이 노트북 하나에서 이어서** 해요.

## 시작하기

1. 위 메뉴에서 **파일 → Drive에 사본 저장**을 눌러요. 내 사본에 답을 남길 수 있어요.
2. 첫 코드 칸 왼쪽의 **▶**를 눌러요. `준비 완료`가 나오면 다음 칸으로 가요.
3. 수업 중에는 안내한 칸만 실행해요. **Shift+Enter**도 같은 칸을 실행하는 방법이에요.

`Cell`은 코드나 설명이 들어 있는 칸이에요. 런타임이 초기화되어 파일이나 변수가 사라졌다면 첫 칸부터 다시 실행해요. 단순히 브라우저를 다시 여는 것과는 달라요.

코드 앞의 `#`는 설명이에요. 실행되지 않아요. `import`는 다른 파일의 이름을 가져오고, 줄 앞의 `!`는 Python 대신 터미널 명령을 실행해요.


In [ ]:
# 필요한 파일을 Colab으로 받아 와요. 실행 상태가 초기화되면 다시 준비해요.
# import는 이름을 가져오고, !는 터미널 명령을 실행하는 표시예요.
import os, sys

if not os.path.isdir("jnu-llmops-precourse-day2"):
    !git clone -q https://github.com/GoBeromsu/jnu-llmops-precourse-day2.git
if not os.path.isdir("jnu-llmops-precourse-day3"):
    !git clone -q https://github.com/GoBeromsu/jnu-llmops-precourse-day3.git

# 내려받기가 실패했는데 준비됐다고 표시하지 않아요.
for filename in (
    "jnu-llmops-precourse-day2/solution/catalog.py",
    "jnu-llmops-precourse-day2/solution/order.py",
    "jnu-llmops-precourse-day2/solution/pricing.py",
    "jnu-llmops-precourse-day2/solution/receipt.py",
    "jnu-llmops-precourse-day3/data/orders.json",
    "jnu-llmops-precourse-day3/data/expected_day3.json",
    "jnu-llmops-precourse-day3/data/expected_day4.json",
):
    if not os.path.isfile(filename):
        raise FileNotFoundError(f"준비 파일이 없어요: {filename}. 위 다운로드 오류를 강사에게 보여 주세요.")

sys.path.insert(0, "jnu-llmops-precourse-day2/solution")
print("준비 완료 · 왼쪽 파일 탭에 두 폴더가 생겼는지 확인하세요")


## 버전 A · 어제 만든 파일을 불러요

다음 칸을 실행하기 전에 최종 금액을 예상해 보세요. 실행하면 영수증이 나와요.

`Order`는 주문을 담고, `calculate_bill`은 금액을 계산해요. `format_receipt`는 영수증 글자를 만들어요. `from 파일이름 import 이름`은 그 파일의 이름을 여기서 쓰겠다는 뜻이에요.

In [ ]:
# 어제 완료본의 네 파일에서 이름 넷을 꺼내 옵니다.
# from 파일이름 import 이름  = "그 파일 안의 이 이름을 여기서 쓰겠다"
from catalog import MENU               # 메뉴판 (이름 → 가격)
from order import Order                # 주문 상자를 만드는 틀
from pricing import calculate_bill     # 주문 → 합계·할인·최종 금액
from receipt import format_receipt     # 주문 + 금액 → 영수증 문자열

order_a = Order()                                    # 빈 주문 하나
order_a.add("카페라떼", 2, MENU["카페라떼"])          # 항목 추가 (메뉴, 수량, 단가)
bill_a = calculate_bill(order_a, True)               # True = 학생 할인
print(format_receipt(order_a, bill_a))               # 영수증을 화면에 찍는다
total_a = bill_a.total                               # 최종 금액을 이름에 남겨 둔다

## 버전 B · 항목을 묶음으로 담아요

다음 칸은 `(메뉴, 수량, 단가)`를 한 묶음으로 저장해요. 계산은 이 칸에서 해요. 실행하면 수량과 최종 금액이 나와요.

In [ ]:
class OrderB:
    def __init__(self):
        self.items = []                      # (메뉴, 수량, 단가)
    def add(self, menu_name, quantity, unit_price):
        self.items.append((menu_name, quantity, unit_price))

order_b = OrderB()
order_b.add("카페라떼", 2, 4000)
subtotal_b = sum(q * p for _, q, p in order_b.items)
total_b = subtotal_b - subtotal_b * 10 // 100      # 학생 할인 10%
print("수량:", order_b.items[0][1])
print("최종 금액:", total_b, "원")

## 버전 C · Cafe 하나에 담아요

메뉴판·주문·계산·영수증을 한 Class에 넣었어요. 실행한 뒤 A·B와 최종 금액을 비교해 보세요.

In [ ]:
class Cafe:
    MENU = {"아메리카노": 3000, "카페라떼": 4000, "초코라떼": 4500}
    def __init__(self):
        self.items = []                      # (메뉴, 수량)
    def add(self, menu_name, quantity):
        self.items.append((menu_name, quantity))
    def total(self, is_student):
        subtotal = sum(q * self.MENU[m] for m, q in self.items)
        discount = subtotal * 10 // 100 if is_student else 0
        return subtotal - discount
    def receipt(self, is_student):
        lines = [f"{m} x {q}" for m, q in self.items]
        lines.append(f"최종 금액: {self.total(is_student)}원")
        return "\n".join(lines)

cafe = Cafe()
cafe.add("카페라떼", 2)
print(cafe.receipt(True))
total_c = cafe.total(True)

## 세 금액을 비교해요

다음 칸을 실행하면 A·B·C의 금액이 나란히 나와요. 모두 7,200원이면 검사를 통과해요.

`AssertionError`가 나오면 세 금액 중 다른 값을 먼저 찾으세요. 그 버전의 입력과 계산 줄을 확인한 뒤 다시 실행해요.

In [ ]:
print("A 네 겹   :", total_a)
print("B 두 겹   :", total_b)
print("C 한 덩어리:", total_c)
assert total_a == total_b == total_c == 7200
print("세 구조, 같은 결과")

## 찾은 위치를 적어요

이 설명 칸을 두 번 누르면 답을 적을 수 있어요. 처음에는 할인율 질문 하나부터 답해 보세요.

1. 학생 할인을 10%에서 15%로 바꾸려면 A·B·C의 어느 줄을 고치나요?
2. 수량 0을 넣으면 각 버전은 어디서 멈추나요? 멈추지 않는 버전도 있나요?
3. 계산만 따로 확인하고 싶다면 어느 버전이 편한가요? 이유는 무엇인가요?

내 답:
- 할인율을 바꿀 위치: A / B / C
- 수량 0을 넣은 결과:
- 따로 확인하기 편한 버전과 이유:

코드를 아직 다 읽지 못해도 괜찮아요. 찾은 줄을 먼저 적고, 남은 질문을 강사에게 보여 주세요.